In [30]:
import numpy as np 
import matplotlib.pyplot as plt
import OrcFxAPI
import h5py
import pandas as pd
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.optimize import minimize


In [31]:
model_path = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Decay_opzet_spring_fine_V5.dat"
sim_path = r"c:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Sim_data\Masscheck_pitch_V1"

lookback_window = 5.0      # seconden vóór decay-start waarin we de amplitude zoeken
quiet_window_end = 12.0     # rustige periode eindigt 12 s vóór decay-start
quiet_window_length = 50.0


In [32]:
model = OrcFxAPI.Model(model_path)
constraint = model["decay_constraint"]
floaters = model["floaters"]
floatertype = model["Floatertype"]

In [33]:
constraint.InFrameInitialZ = 0
constraint.InFrameInitialX = 0
constraint.InFrameInitialY = 0
constraint.InFrameInitialAzimuth = 0
floaters.InitialTrim = 4
constraint.InFrameInitialGamma = 0

floatertype.OtherDampingLinearCoeffx = 0 #surge
floatertype.OtherDampingLinearCoeffy = 0 #sway
floatertype.OtherDampingLinearCoeffz  = 0 #heav
floatertype.OtherDampingLinearCoeffRx = 0 #roll
floatertype.OtherDampingLinearCoeffRy = 0 #pitch
floatertype.OtherDampingLinearCoeffRz = 0 #yaw

floatertype.OtherDampingQuadraticCoeffx = 0 #surge
floatertype.OtherDampingQuadraticCoeffy = 0 #sway 
floatertype.OtherDampingQuadraticCoeffz = 0 #heav
floatertype.OtherDampingQuadraticCoeffRx = 0#roll
floatertype.OtherDampingQuadraticCoeffRy = 0 #pitch
floatertype.OtherDampingQuadraticCoeffRz = 0 #yaw


In [34]:
model.RunSimulation()
print("Simulatie klaar.")

Simulatie klaar.


In [35]:
def compute_period(t, z):
    peaks, _ = find_peaks(z)
    
    if len(peaks) < 2:
        return None  # geen betrouwbare periode
    
    T = np.diff(t[peaks])
    return np.mean(T)

In [36]:
target_T = 7.26

mass_values = np.linspace(75,77, 5)  # kies range die logisch is
results = []

for i, mass in enumerate(mass_values):

    model = OrcFxAPI.Model(model_path)
    
    constraint = model["decay_constraint"]
    floaters = model["floaters"]
    floatertype = model["Floatertype"]

    floaters.InitialTrim = 4

    floatertype.Mass = mass

    floatertype.OtherDampingLinearCoeffRy = 0
    floatertype.OtherDampingQuadraticCoeffRy = 0

    model.RunSimulation()

    model.SaveSimulation(f"{sim_path}\{i}_V1.dat")

    print(f"Sim {i} klaar (mass={mass})")

    

<string>:23: SyntaxWarning: invalid escape sequence '\{'
<>:23: SyntaxWarning: invalid escape sequence '\{'
<string>:23: SyntaxWarning: invalid escape sequence '\{'
<>:23: SyntaxWarning: invalid escape sequence '\{'
C:\Users\verav\AppData\Local\Temp\ipykernel_7244\3509641732.py:23: SyntaxWarning: invalid escape sequence '\{'
  model.SaveSimulation(f"{sim_path}\{i}_V1.dat")


Sim 0 klaar (mass=75.0)
Sim 1 klaar (mass=75.5)
Sim 2 klaar (mass=76.0)
Sim 3 klaar (mass=76.5)
Sim 4 klaar (mass=77.0)


In [37]:
results = []

for i, mass in enumerate(mass_values):

    model = OrcFxAPI.Model(f"{sim_path}\{i}_V1.dat")
    floaters = model["floaters"]

    t_sim = model.general.TimeHistory("Time")
    z_sim = floaters.TimeHistory("Rotation 2")

    T = compute_period(t_sim, z_sim)

    results.append((mass, T));

<string>:5: SyntaxWarning: invalid escape sequence '\{'
<>:5: SyntaxWarning: invalid escape sequence '\{'
<string>:5: SyntaxWarning: invalid escape sequence '\{'
<>:5: SyntaxWarning: invalid escape sequence '\{'
C:\Users\verav\AppData\Local\Temp\ipykernel_7244\1510069692.py:5: SyntaxWarning: invalid escape sequence '\{'
  model = OrcFxAPI.Model(f"{sim_path}\{i}_V1.dat")


In [38]:
best = min(results, key=lambda x: abs(x[1] - target_T))

print("Beste resultaat:")
print(f"massa = {best[0]:.6f}")
print(f"periode = {best[1]:.6f}")
print(f"afwijking = {abs(best[1] - target_T):.6f}")


Beste resultaat:
massa = 77.000000
periode = 7.180000
afwijking = 0.080000
